In [1]:
!pip install pandas scikit-learn torch transformers "datasets<2.19.0" evaluate tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 17.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.2.0 which is incompatible.


In [2]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=5f487cc40b486513807cedd4790b6511e1af54aa4619514eed9ecfd9ecd82480
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [3]:

print("--- Step 1: Installing and Importing libraries ---")
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import T5ForConditionalGeneration, T5Tokenizer, get_linear_schedule_with_warmup
from tqdm.auto import tqdm
import evaluate
from sentence_transformers import SentenceTransformer, util

--- Step 1: Installing and Importing libraries ---


In [4]:


print("\n--- Step 2: Loading and preprocessing data ---")

df = pd.read_csv('mtsamples.csv')


df = df[['transcription', 'keywords']].dropna().reset_index(drop=True)


initial_rows = len(df)
df.drop_duplicates(subset=['transcription', 'keywords'], inplace=True)
df.reset_index(drop=True, inplace=True)
rows_removed = initial_rows - len(df)
print(f"Removed {rows_removed} duplicate records. New dataset shape: {df.shape}")

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_keywords(keywords):
    keywords = str(keywords).replace(',', ' ').replace(';', ' ')
    keywords = re.sub(r'\s+', ' ', keywords).strip()
    return keywords

df['transcription'] = df['transcription'].apply(clean_text)
df['keywords'] = df['keywords'].apply(clean_keywords)
df = df[df['keywords'].apply(len) > 0]
df['transcription'] = "keywords: " + df['transcription']


sample_df = df

train_df, test_df = train_test_split(sample_df, test_size=0.2, random_state=42)
print(f"Training on {len(train_df)} records, testing on {len(test_df)} records.")


--- Step 2: Loading and preprocessing data ---
Removed 46 duplicate records. New dataset shape: (3852, 2)
Training on 3053 records, testing on 764 records.


In [5]:

MODEL_NAME = 't5-base' # Using the more powerful 'base' model
# =======================================================================
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME, model_max_length=512)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Using device: {device}")

class KeywordDataset(Dataset):
    def __init__(self, dataframe, tokenizer, source_max_len=512, target_max_len=128):
        self.tokenizer = tokenizer
        self.data = dataframe
        self.source_max_len = source_max_len
        self.target_max_len = target_max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        source_text = self.data.iloc[index]['transcription']
        target_text = self.data.iloc[index]['keywords']
        source = self.tokenizer.batch_encode_plus([source_text], max_length=self.source_max_len, padding='max_length', truncation=True, return_tensors='pt')
        target = self.tokenizer.batch_encode_plus([target_text], max_length=self.target_max_len, padding='max_length', truncation=True, return_tensors='pt')
        source_ids = source['input_ids'].squeeze()
        source_mask = source['attention_mask'].squeeze()
        target_ids = target['input_ids'].squeeze()
        labels = target_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {'source_ids': source_ids.to(dtype=torch.long), 'source_mask': source_mask.to(dtype=torch.long), 'labels': labels.to(dtype=torch.long)}

train_dataset = KeywordDataset(train_df, tokenizer)
test_dataset = KeywordDataset(test_df, tokenizer)

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Using device: cuda


In [6]:

print("\n--- Step 4: Starting model fine-tuning ---")
EPOCHS = 6
BATCH_SIZE = 4 # Kept small for larger model
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
optimizer = AdamW(model.parameters(), lr=1e-4) # Slightly lower learning rate for bigger model

for epoch in range(EPOCHS):
    print(f"--- Epoch {epoch+1}/{EPOCHS} ---")
    model.train()
    progress_bar = tqdm(train_loader, desc="Training")
    for batch in progress_bar:
        optimizer.zero_grad()
        outputs = model(input_ids=batch['source_ids'].to(device), attention_mask=batch['source_mask'].to(device), labels=batch['labels'].to(device))
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        progress_bar.set_postfix({'loss': loss.item()})



--- Step 4: Starting model fine-tuning ---
--- Epoch 1/6 ---


Training:   0%|          | 0/764 [00:00<?, ?it/s]

--- Epoch 2/6 ---


Training:   0%|          | 0/764 [00:00<?, ?it/s]

--- Epoch 3/6 ---


...

Training:   0%|          | 0/764 [00:00<?, ?it/s]

--- Epoch 5/6 ---


Training:   0%|          | 0/764 [00:00<?, ?it/s]

--- Epoch 6/6 ---


Training:   0%|          | 0/764 [00:00<?, ?it/s]

In [7]:

print("\n--- Step 5: Evaluating the fine-tuned model ---")
model.eval()
predictions = []
actuals = []
rouge_scorer = evaluate.load('rouge')
print("Loading model for Semantic Similarity calculation...")
semantic_model = SentenceTransformer('all-MiniLM-L6-v2')
semantic_model.to(device)

with torch.no_grad():
    for index in tqdm(range(len(test_df)), desc="Generating Predictions"):
        row = test_df.iloc[index]
        text = row['transcription']
        true_keywords = row['keywords']
        encoding = tokenizer.encode_plus(text, max_length=512, padding='max_length', truncation=True, return_tensors="pt")
        input_ids = encoding['input_ids'].to(device)
        attention_mask = encoding['attention_mask'].to(device)
        generated_ids = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_length=128, num_beams=4, repetition_penalty=2.5, length_penalty=1.0, early_stopping=True)
        pred = tokenizer.decode(generated_ids[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)
        predictions.append(pred)
        actuals.append(true_keywords)

rouge_results = rouge_scorer.compute(predictions=predictions, references=actuals)

print("Calculating Semantic Similarity...")
semantic_scores = []
with torch.no_grad():
    for pred, actual in tqdm(zip(predictions, actuals), total=len(predictions), desc="Semantic Similarity"):
        if not pred or not actual:
            semantic_scores.append(0.0)
            continue
        pred_emb = semantic_model.encode(pred, convert_to_tensor=True)
        actual_emb = semantic_model.encode(actual, convert_to_tensor=True)
        score = util.cos_sim(pred_emb, actual_emb).item()
        semantic_scores.append(score)
avg_semantic_score = np.mean(semantic_scores)

print("\n\n" + "="*25 + " FINAL RESULTS " + "="*25)
print("Fine-Tuned T5 Model Performance on Test Set")
print(f"ROUGE-1 (F1): {rouge_results['rouge1']:.4f}")
print(f"ROUGE-L (F1): {rouge_results['rougeL']:.4f}")
print(f"Semantic Similarity: {avg_semantic_score:.4f}")
print("="*67)

print("\n--- Example Predictions ---")
for i in range(min(5, len(predictions))):
    print(f"\n--- Example {i+1} ---")
    print(f"TRANSCRIPTION (snippet): {test_df.iloc[i]['transcription'][10:210]}...")
    print(f"TRUE KEYWORDS:     {actuals[i]}")
    print(f"T5 PREDICTION:     {predictions[i]}")
    print("-" * 20)

print("\n--- Pipeline Finished ---")


--- Step 5: Evaluating the fine-tuned model ---


Loading model for Semantic Similarity calculation...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating Predictions:   0%|          | 0/764 [00:00<?, ?it/s]

Calculating Semantic Similarity...


Semantic Similarity:   0%|          | 0/764 [00:00<?, ?it/s]



========================= FINAL RESULTS =========================
Fine-Tuned T5 Model Performance on Test Set
ROUGE-1 (F1): 0.6223
ROUGE-L (F1): 0.5244
Semantic Similarity: 0.8180

--- Example Predictions ---

--- Example 1 ---
TRANSCRIPTION (snippet): preoperative diagnosis: , chronic renal failure.,postoperative diagnosis: ,chronic renal failure.,procedure performed:, insertion of left femoral circle-c catheter.,anesthesia: , 1% lidocaine.,estimat...
TRUE KEYWORDS:     nephrology chronic renal failure femoral circle-c catheter indwelling catheter catheter insertion seldinger guidewire indwelling femoral dialysis
T5 PREDICTION:     nephrology chronic renal failure insertion of left femoral circle-c catheter indwelling catheter seldinger guidewire femoral vein dialysis catheter
--------------------

--- Example 2 ---
TRANSCRIPTION (snippet): findings:,normal foramen magnum.,normal brainstem-cervical cord junction. there is no tonsillar ectopia. normal clivus and craniovertebral junct